<a href="https://colab.research.google.com/github/lmassaron/fine-tuning-workshop/blob/main/reasoned-financial-sentiment/03_sentiment_evaluation%20(b).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Evaluation of Fine-Tuned Sentiment Analysis Model (Gemma 4 E2B-IT CoT)

This notebook provides a comprehensive evaluation of the **Gemma 4 E2B-IT** model after it has been fine-tuned for reasoned financial sentiment analysis (Chain-of-Thought).

### Why Multi-Dimensional Evaluation?
A simple accuracy score is often insufficient for Large Language Models. To truly understand the model's performance, we evaluate it across three critical dimensions:
1. **Sentiment Accuracy**: Does the predicted label (Positive/Neutral/Negative) match the ground truth on the clean, non-augmented test set?
2. **Tag Integrity (Instruction Following)**: Did the model correctly use the requested `<sentiment>` and `<reasoning>` tags? This is vital for parsing the output in automated pipelines.
3. **Reasoning Quality (LLM-as-a-judge)**: How insightful and accurate are the model's explanations? We use a larger model (**Qwen 2.5 7B**) to grade these explanations on a scale of 1–10.

### Hardware Verification

Before starting, we check the GPU status using `nvidia-smi`.

| Metric | Description |
|---|---|
| `GPU-Util` | Percentage of GPU cores currently active. |
| `Memory-Usage` | Amount of VRAM occupied. Loading our model in 4-bit will typically use ~3–5GB. |
| `Perf` | Performance state (P0 is max performance). |

In [ ]:
import platform

if platform.system() != "Darwin":
    # Check the GPU information
    !nvidia-smi

### Environment Setup

We install and import the necessary libraries. We'll use `peft` to load our LoRA adapter and `sklearn` for calculating traditional classification metrics.

In [ ]:
%%capture
# Install necessary libraries for model evaluation
import os

if "COLAB_" not in "".join(os.environ.keys()):
    pass
else:
    !pip install -U transformers trl peft accelerate bitsandbytes datasets

In [ ]:
# Suppress non-critical warnings to keep notebook output clean
import warnings

warnings.filterwarnings("ignore")

In [ ]:
# Numeric and data utilities
import gc
import os
import re  # regular expressions for parsing model output tags
import pandas as pd
from tqdm import tqdm

# PyTorch
import torch

# Hugging Face Transformers
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,  # 4-bit quantization configuration
)

from peft import PeftConfig, PeftModel

# Hugging Face ecosystem
from datasets import load_dataset
from huggingface_hub import login

# Evaluation metrics
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1. Configuration, Dataset & Model Loading

We configure our evaluation paths, load the clean (non-augmented) test split from `lmassaron/FinancialPhraseBank_explained`, and load our fine-tuned **Gemma 4 E2B-IT** model using `AutoModelForCausalLM` in 4-bit quantization with the **LoRA adapter** attached.

> **Optional Fast Path:** If you already ran the integrated training notebook and exported `test_predictions_with_reasoning.csv`, set `USE_PRECOMPUTED_CSV = True` below to skip re-running inference and jump straight to the metrics and the Qwen 2.5 7B judge.

In [ ]:
def define_device():
    """Determines and returns the optimal PyTorch device based on hardware availability."""
    print(f"PyTorch version: {torch.__version__}", end=" -- ")

    # Prefer MPS (Metal Performance Shaders) on Apple Silicon Macs
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        print("using MPS device on macOS")
        return torch.device("mps")

    # Fall back to CUDA (NVIDIA GPU) or CPU
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"using {device}")
    return device


device = define_device()
print(f"Device: {device}")

In [ ]:
def init():
    """Sets environment variables, logs in to Hugging Face, and clears GPU memory."""
    os.environ["TOKENIZERS_PARALLELISM"] = "false"  # avoid deadlocks in forked tokenizer processes
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"  # pin to the first GPU

    token = os.environ.get("HF_TOKEN")
    if token:
        login(token=token)
    else:
        try:
            from google.colab import userdata

            # Read the token stored in Colab's Secrets tab
            hf_token = userdata.get("HF_TOKEN")
            if hf_token:
                login(token=hf_token, add_to_git_credential=True)
                print("Hugging Face login successful using Google Colab secrets!")
            else:
                print("Error: HF_TOKEN not found in Google Colab secrets or is empty.")
                print("Please ensure you have created a secret named 'HF_TOKEN' in the 'Secrets' tab.")
        except Exception:
            print("HF_TOKEN not set. You might need to log in manually.")

    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # release any stale GPU allocations from a previous run
    gc.collect()
    warnings.filterwarnings("ignore")


init()

In [ ]:
# Use the local adapter directory ("./gemma4-sentiment-lora-adapter")
# or your Hugging Face Hub repo for the fine-tuned Gemma 4 E2B-IT adapter
MODEL_PATH = "lmassaron/qlora4b-Gemma4-E2B-finsent-cot"
DATASET_ID = "lmassaron/FinancialPhraseBank_explained"

# Optional fast path: set to True to reuse CSV predictions from the training notebook
USE_PRECOMPUTED_CSV = False
PRECOMPUTED_CSV_PATH = "test_predictions_with_reasoning.csv"

# Shared 4-bit quantization config (used for both Gemma 4 and the Qwen 2.5 7B judge)
compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)
QUANTIZATION_CONFIG = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print("Loading test dataset...")
ds = load_dataset(DATASET_ID, split="test")
test_df = ds.to_pandas()

# Filter out augmented rows so evaluation matches the 484 original benchmark headlines
if "is_augmented" in test_df.columns:
    test_df = test_df[test_df["is_augmented"] == False].reset_index(drop=True)

print(f"Clean test headlines loaded: {len(test_df)}")

In [ ]:
if not USE_PRECOMPUTED_CSV:
    # 1. Read adapter config to get the base model id (e.g. "google/gemma-4-E2B-it")
    peft_config = PeftConfig.from_pretrained(MODEL_PATH)
    BASE_MODEL_ID = peft_config.base_model_name_or_path

    # 2. Load tokenizer with left-padding for batched decoder-only generation
    print(f"Loading model from {MODEL_PATH} (base: {BASE_MODEL_ID})...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # 3. Load the quantized base model via AutoModelForCausalLM (supports Gemma 4)
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        dtype=compute_dtype,
        quantization_config=QUANTIZATION_CONFIG,
        device_map="auto",
    )

    # 4. Attach the LoRA adapter on top
    model = PeftModel.from_pretrained(base_model, MODEL_PATH)
    model = model.eval()
    print("Model and adapter ready for evaluation.")
else:
    print(f"USE_PRECOMPUTED_CSV=True — skipping model load; will read {PRECOMPUTED_CSV_PATH}.")

## 2. Batch Inference

We run inference on the clean test set (or load the precomputed CSV if `USE_PRECOMPUTED_CSV = True`).

Crucially, we use the **exact same `INSTRUCTION` prompt** and headline format that the model saw during fine-tuning. We capture the **raw output** and use regular expressions (`re`) to extract `<sentiment>` and `<reasoning>`.

In [ ]:
INSTRUCTION = (
    "You are a financial analyst with expertise in equity markets and corporate finance.\n"
    "Analyze the following financial news headline and determine its market sentiment "
    "from an investor's perspective.\n\n"
    "Classify the sentiment as positive, neutral, or negative based on the likely "
    "impact on stock price, investor confidence, or financial performance.\n\n"
    "Respond using exactly these two tags:\n"
    "<sentiment>positive|neutral|negative</sentiment>\n"
    "<reasoning>brief financial explanation</reasoning>\n"
)


def extract_tag(text, tag):
    match = re.search(rf"<{tag}>(.*?)</{tag}>", text, re.IGNORECASE | re.DOTALL)
    return match.group(1).strip() if match else None


if USE_PRECOMPUTED_CSV:
    raw_csv = pd.read_csv(PRECOMPUTED_CSV_PATH)
    results_df = pd.DataFrame(
        {
            "sentence": raw_csv["headline"],
            "sentiment_true": raw_csv["y_true"],
            "raw_output": "",
            "sentiment_pred": raw_csv["y_pred"].fillna("none"),
            "reasoning_pred": raw_csv["reasoning"].fillna("none"),
            "sentiment_tag_present": raw_csv["tag_integrity"].astype(bool),
            "reasoning_tag_present": raw_csv["tag_integrity"].astype(bool),
        }
    )
    print(f"Loaded {len(results_df)} precomputed predictions from {PRECOMPUTED_CSV_PATH}.")
else:
    results = []
    batch_size = 16

    for start in tqdm(range(0, len(test_df), batch_size), desc="Evaluating"):
        batch_df = test_df.iloc[start : start + batch_size]

        prompts = []
        for _, row in batch_df.iterrows():
            # Matches the exact training prompt format (no trailing newline after the quote)
            messages = [
                {
                    "role": "user",
                    "content": INSTRUCTION + f'\nHeadline: "{row["sentence"]}"',
                }
            ]
            prompts.append(
                tokenizer.apply_chat_template(
                    messages, tokenize=False, add_generation_prompt=True
                )
            )

        inputs = tokenizer(
            prompts, return_tensors="pt", padding=True, truncation=True, max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=200,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )

        input_len = inputs["input_ids"].shape[1]
        decoded = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)

        for i, raw_output in enumerate(decoded):
            sentiment_tag = extract_tag(raw_output, "sentiment")
            reasoning_pred = extract_tag(raw_output, "reasoning")

            # Robust extraction with keyword fallback
            sentiment_text = sentiment_tag.lower() if sentiment_tag else raw_output.lower()
            if "positive" in sentiment_text:
                sentiment_pred = "positive"
            elif "negative" in sentiment_text:
                sentiment_pred = "negative"
            elif "neutral" in sentiment_text:
                sentiment_pred = "neutral"
            else:
                sentiment_pred = "none"

            results.append(
                {
                    "sentence": batch_df.iloc[i]["sentence"],
                    "sentiment_true": batch_df.iloc[i]["sentiment"],
                    "raw_output": raw_output,
                    "sentiment_pred": sentiment_pred,
                    "reasoning_pred": reasoning_pred if reasoning_pred else "none",
                    "sentiment_tag_present": sentiment_tag is not None,
                    "reasoning_tag_present": reasoning_pred is not None,
                }
            )

    results_df = pd.DataFrame(results)

## 3. Metric 1: Sentiment Accuracy

We use `classification_report` and `confusion_matrix` to see how well the model performs for each category.

**Key Metrics:**
- **Precision**: When the model predicts "Positive", how often is it actually positive?
- **Recall**: Out of all actual "Positive" headlines, how many did the model find?
- **F1-Score**: The harmonic mean of precision and recall.

In [ ]:
accuracy = accuracy_score(results_df["sentiment_true"], results_df["sentiment_pred"])
print(f"Overall Sentiment Accuracy: {accuracy:.2%}")
print("\nClassification Report:")
print(classification_report(results_df["sentiment_true"], results_df["sentiment_pred"]))

labels = ["negative", "neutral", "positive"]
print("Confusion Matrix (rows=true, cols=pred [neg, neu, pos]):")
print(confusion_matrix(results_df["sentiment_true"], results_df["sentiment_pred"], labels=labels))

## 4. Metric 2: Tag Integrity

This measures instruction following. If the model fails to output `<sentiment>` or `<reasoning>` tags, it will break downstream processing. A score of 100% is the goal for production reliability.

In [ ]:
sentiment_tag_rate = results_df["sentiment_tag_present"].mean()
reasoning_tag_rate = results_df["reasoning_tag_present"].mean()
both_tags_rate = (
    results_df["sentiment_tag_present"] & results_df["reasoning_tag_present"]
).mean()

print(f"<sentiment> tag present: {sentiment_tag_rate:.2%}")
print(f"<reasoning> tag present: {reasoning_tag_rate:.2%}")
print(f"Both tags correctly formed: {both_tags_rate:.2%}")

## 5. Metric 3: Reasoning Quality (LLM-as-a-judge)

How do we know if the reasoning is actually "good"? We use a larger, more capable model (**Qwen 2.5 7B**) as an impartial judge.

### Evaluation Rubric (1–10)
| Score | Meaning |
|-------|---------|
| 1–2   | Wrong, irrelevant, or missing reasoning. |
| 3–4   | Partially correct but shallow or off-topic. |
| 5–6   | Accurate but generic; no link to price, confidence, or performance. |
| 7–8   | Solid and relevant; connects the headline to a financial outcome. |
| 9–10  | Precise, insightful, and concise; clearly justifies the sentiment. |

Before loading the 7B judge, we explicitly delete both `model` and `base_model` and clear the CUDA cache so all GPU VRAM is reclaimed.

In [ ]:
# Release both the PEFT wrapper and the underlying base model to reclaim GPU VRAM
for var_name in ["model", "base_model"]:
    if var_name in globals():
        del globals()[var_name]
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
print(f"Loading Judge Model: {JUDGE_MODEL_ID}...")
judge_tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL_ID)
judge_model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL_ID,
    dtype=compute_dtype,
    quantization_config=QUANTIZATION_CONFIG,
    device_map="auto",
)

JUDGE_PROMPT = (
    "You are a master financial analyst judging the quality of reasoning for a sentiment classification.\n\n"
    'Headline: "{sentence}"\n'
    "Classified Sentiment: {sentiment}\n"
    'Provided Reasoning: "{reasoning}"\n\n'
    "Score the reasoning 1-10 across financial accuracy, depth of insight, and clarity:\n"
    "  1-2  Wrong, irrelevant, or missing.\n"
    "  3-4  Partially correct but shallow or off-topic.\n"
    "  5-6  Accurate but generic; no link to price, confidence, or performance.\n"
    "  7-8  Solid and relevant; connects the headline to a financial outcome.\n"
    "  9-10 Precise, insightful, and concise; clearly justifies the sentiment.\n\n"
    "Respond with a single number from 1 to 10. Do not provide any other text."
)

BATCH_SIZE = 8
scores = []
rows = results_df.to_dict("records")

if judge_tokenizer.pad_token is None:
    judge_tokenizer.pad_token = judge_tokenizer.eos_token

for batch_start in tqdm(range(0, len(rows), BATCH_SIZE), desc="Judging Reasoning"):
    batch = rows[batch_start : batch_start + BATCH_SIZE]

    valid_prompts = []
    valid_indices = []
    batch_scores = [1] * len(batch)  # Default to 1 for "none" reasoning cases

    for i, row in enumerate(batch):
        if str(row["reasoning_pred"]).lower() != "none":
            prompt = JUDGE_PROMPT.format(
                sentence=row["sentence"],
                sentiment=row["sentiment_pred"],
                reasoning=row["reasoning_pred"],
            )
            valid_prompts.append([{"role": "user", "content": prompt}])
            valid_indices.append(i)

    if not valid_prompts:
        scores.extend(batch_scores)
        continue

    judge_tokenizer.padding_side = "left"  # Required for decoder-only generation
    inputs = judge_tokenizer.apply_chat_template(
        valid_prompts,
        tokenize=True,
        add_generation_prompt=True,
        padding=True,
        return_tensors="pt",
        return_dict=True,
    ).to(judge_model.device)

    with torch.no_grad():
        outputs = judge_model.generate(
            **inputs,
            max_new_tokens=5,
            do_sample=False,
            pad_token_id=judge_tokenizer.pad_token_id,
        )

    prompt_len = inputs["input_ids"].shape[1]

    for local_idx, original_idx in enumerate(valid_indices):
        generated_tokens = outputs[local_idx][prompt_len:]
        decoded = judge_tokenizer.decode(
            generated_tokens, skip_special_tokens=True
        ).strip()

        match = re.search(r"\b([1-9]|10)\b", decoded)
        if match:
            batch_scores[original_idx] = int(match.group(1))
        else:
            batch_scores[original_idx] = pd.NA

    scores.extend(batch_scores)

results_df["reasoning_score"] = pd.array(scores, dtype="Int64")
avg_score = results_df["reasoning_score"].dropna().mean()
print(f"\nAverage Reasoning Quality Score: {avg_score:.2f} / 10")

# Export complete evaluation results (sentiment, tags, reasoning, and judge score)
results_df.to_csv("evaluation_results_with_judge_scores.csv", index=False)
print("Saved full evaluation results to evaluation_results_with_judge_scores.csv")